In [ ]:
# !pip install fusion_solar_py -q

In [1]:
# from kaggle_secrets import UserSecretsClient

# user_secrets = UserSecretsClient()
# FUSION_SOLAR_CLIENT_PASSWORD = user_secrets.get_secret("FUSION_SOLAR_CLIENT_PASSWORD")
# FUSION_SOLAR_CLIENT_USERNAME = user_secrets.get_secret("FUSION_SOLAR_CLIENT_USERNAME")

In [4]:
from dotenv import load_dotenv
import os

load_dotenv()
FUSION_SOLAR_CLIENT_PASSWORD = os.environ.get("FUSION_SOLAR_CLIENT_PASSWORD")
FUSION_SOLAR_CLIENT_USERNAME = os.environ.get("FUSION_SOLAR_CLIENT_USERNAME")

In [5]:
import time
from datetime import datetime
from datetime import timezone

import numpy as np
import pandas as pd
from tqdm import tqdm

from energymanagementrl.fusion_solar_extension import FusionSolarClientExtended

In [6]:
# log into the API - with proper credentials...
client = FusionSolarClientExtended(FUSION_SOLAR_CLIENT_USERNAME, FUSION_SOLAR_CLIENT_PASSWORD,
                                   huawei_subdomain="uni004eu5")
_plant_id = client.get_plant_ids()[0]
_battery_id = client.get_battery_ids(_plant_id)[0]

In [22]:
client.set_battery_working_mode(_battery_id, client.BatteryWorkingMode.MAXIMUM_SELF_CONSUMPTION)

In [189]:
plant_data = client.get_plant_stats(_plant_id)

In [ ]:
client.get_battery_day_stats(_battery_id)

In [ ]:
# Constants
DAY_IN_MILLIS = 24 * 60 * 60 * 1000  # Milliseconds in a day
START_TODAY = time.strftime("%Y-%m-%d 00:00:00", time.gmtime())
START_TODAY_MILLIS = int(
    datetime.strptime(START_TODAY, "%Y-%m-%d %H:%M:%S").replace(tzinfo=timezone.utc).timestamp()) * 1000
keys_to_keep = ['xAxis', "productPower", 'usePower', 'selfUsePower', 'chargePower', 'dischargePower']
partials = []
days = (datetime.now() - datetime.strptime(client.get_plant_details(_plant_id)['gridConnectedTime'].split()[0],
                                           '%Y-%m-%d')).days
for day_offset in tqdm(range(days)):
    # Calculate the timestamp for the target day
    target_timestamp_millis = START_TODAY_MILLIS - (DAY_IN_MILLIS * day_offset)
    # Fetch battery stats for the given timestamp
    battery_stats = client.get_battery_day_stats(
        _battery_id,
        target_timestamp_millis,
        signals=[client.BatterySignal.SOC],
    )
    
    # Convert each battery element to a DataFrame
    battery_stats_dfs = [
        pd.DataFrame(element['pmDataList'])
        .loc[:, ['startTime', 'counterValue']]
        .rename(columns={'counterValue': element['name']})
        .set_index('startTime')
        for element in battery_stats.values()
    ]
    plant_stats = client.get_plant_stats(
        _plant_id,
        target_timestamp_millis,
        time_zone=0.0,
        time_zone_str='Etc/GMT',
    )
    plant_stats_df = pd.DataFrame.from_dict({key: plant_stats[key] for key in keys_to_keep})
    plant_stats_df['startTime'] = pd.to_datetime(plant_stats_df['xAxis'])
    plant_stats_df['startTime'] = plant_stats_df['startTime'].apply(lambda x:x.tz_localize("Europe/Rome", ambiguous=True).timestamp())
    plant_stats_df.drop('xAxis', axis=1, inplace=True)
    plant_stats_df.set_index('startTime', inplace=True)

    full_stats_df = pd.concat(battery_stats_dfs + [plant_stats_df], axis=1)
    full_stats_df = full_stats_df.replace(np.float64(1.7976931348623157e+308), np.nan).replace('--', np.nan).astype(
        np.float16)
    full_stats_df['SOC'] = full_stats_df['SOC']
    to_grid = full_stats_df['productPower'] + full_stats_df['dischargePower'] - full_stats_df['usePower'] - \
              full_stats_df[
                  'chargePower']
    full_stats_df['takenGrid'] = to_grid.apply(lambda x: -x if x < 0 else 0)
    full_stats_df['givenGrid'] = to_grid.apply(lambda x: x if x > 0 else 0)
    full_stats_df = full_stats_df[full_stats_df.index <= pd.Timestamp.now().timestamp()]

    partials.append(full_stats_df)
    

In [28]:
# Concatenate all daily dataframes into the final result
result = pd.concat(partials, axis=0)
result.sort_index(axis=0, inplace=True)
result.info()
result['storePower'] = result['dischargePower'] - result['chargePower']
result['gridPower'] = result['takenGrid'] - result['givenGrid']
result['loadPower'] = -result['usePower']
result['prodPower'] = result['productPower']
plant_stat_history = result[['prodPower', 'loadPower', 'gridPower', 'storePower', 'SOC']]

<class 'pandas.core.frame.DataFrame'>
Index: 73095 entries, 1711922400.0 to 1733850600.0
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   SOC             72935 non-null  float16
 1   productPower    72947 non-null  float16
 2   usePower        72928 non-null  float16
 3   selfUsePower    72928 non-null  float16
 4   chargePower     72947 non-null  float16
 5   dischargePower  72947 non-null  float16
 6   takenGrid       73095 non-null  float64
 7   givenGrid       73095 non-null  float64
dtypes: float16(6), float64(2)
memory usage: 2.5 MB


In [29]:
plant_stat_history.astype(np.float16).to_csv('../data/plant_stat_history.csv')

In [4]:
client.get_parsed_plant_flow(_plant_id)

(1.58, -0.252, 0.64, -1.968, 69.0)